# Week 6: Fixing Models - Generalization and Stability

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rrfhwn/neural-architectures-and-representation-learning-course/blob/main/weeks/06/Week_06_Fixing_Models_Generalization_Stability.ipynb)

**Course:** Neural Architectures and Representation Learning (Master level)

## Learning goals

- Diagnose **underfitting**, **overfitting**, and **unstable training** from train/validation curves.
- Connect model capacity to generalization.
- Use practical PyTorch fixes: dropout, batch normalization, gradient clipping, initialization, and early stopping.
- Compare fixes with controlled experiments instead of guessing.
- Prepare for Assignment 1: repair a broken model and justify the choices.

**Course habit:** diagnose -> change one thing -> run -> observe -> explain.

---

## Environment

**Dependencies:** `torch`, `numpy`, `matplotlib`. CPU is enough.

### Local (uv)

From the repo root:

```bash
uv sync
uv run jupyter notebook weeks/06/Week_06_Fixing_Models_Generalization_Stability.ipynb
```

### Colab

1. Open the notebook via the badge above.
2. Runtime -> Run all.
3. Colab normally includes PyTorch already. Do not upgrade packages unless needed.

In [ ]:
import copy
import math

import numpy as np
import torch
import matplotlib.pyplot as plt

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    plt.rc("axes", grid=True)

%matplotlib inline

print("torch:", torch.__version__)
print("numpy:", np.__version__)

major = int(torch.__version__.split(".")[0])
assert major >= 2, "This notebook expects PyTorch 2.x or newer."

device = torch.device("cpu")
print("device:", device)

---

## 1. Quick recap: what does a broken model look like?

From Weeks 4 and 5:

- **Learning rate too large:** loss jumps, oscillates, or becomes unstable.
- **Underfitting:** train loss and validation loss are both bad.
- **Overfitting:** train loss improves, but validation loss stops improving or gets worse.
- **Early stopping:** use validation loss to decide when to stop.

### Diagnosis table

| Symptom | Likely story | First fix to try |
|---------|--------------|------------------|
| Train bad, val bad | underfitting | increase capacity or train longer |
| Train good, val bad | overfitting | dropout, early stopping, less capacity |
| Loss spikes/noisy | instability | lower lr, clip gradients, improve init |
| Train improves, val plateaus | limited generalization | regularize and validate |

**Pause:** If you only know the final train loss, which failures are invisible?

---

## Repair toolbox overview

Before using the repair tools, name the problem they are meant to solve.

| Tool | Main symptom | Intuition | Tradeoff |
|------|--------------|-----------|----------|
| More capacity | train and validation both bad | give the model enough flexibility to learn the pattern | too much capacity can memorize noise |
| Less capacity | train good, validation bad | make memorization harder | may underfit if reduced too far |
| Dropout | train good, validation bad | randomly hide hidden activations during training so the model cannot rely on one path | train accuracy can look worse |
| Batch normalization | training noisy or slow | keep intermediate activations in a friendlier range | adds moving-statistics behavior to understand |
| Gradient clipping | loss spikes or gradients explode | cap extreme updates before `optimizer.step()` | too much clipping can slow learning |
| Better initialization | training starts unstable or stuck | start weights at a scale suited to the activation | depends on activation and architecture |
| Early stopping | validation stops improving | stop when validation says "enough" | patience must tolerate noisy validation curves |

**Intuition:** each fix should match a symptom. Do not stack tricks before you know what broke.

**Pause and predict:** Which tools might lower training accuracy but improve validation accuracy?

---

## Shared setup: a small nonlinear classification task

We use a synthetic 2D dataset. The label boundary is curved and noisy, so a tiny model can underfit and a large model can overfit.

The task is binary classification:

$$\hat{y} = \sigma(f_\theta(x_1, x_2))$$

where `BCEWithLogitsLoss` combines the sigmoid and binary cross-entropy in a numerically stable way.

In [ ]:
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)


def make_curvy_classification(n_train=80, n_val=400, noise=0.22, seed=0):
    rng = np.random.default_rng(seed)

    def sample(n):
        X = rng.uniform(-2.5, 2.5, size=(n, 2)).astype(np.float32)
        boundary = 0.7 * np.sin(2.2 * X[:, 0]) + 0.25 * X[:, 0]
        noisy_score = X[:, 1] - boundary + rng.normal(0.0, noise, size=n)
        y = (noisy_score > 0).astype(np.float32).reshape(-1, 1)
        return X, y

    X_train, y_train = sample(n_train)
    X_val, y_val = sample(n_val)

    return (
        torch.tensor(X_train, device=device),
        torch.tensor(y_train, device=device),
        torch.tensor(X_val, device=device),
        torch.tensor(y_val, device=device),
    )


X_train, y_train, X_val, y_val = make_curvy_classification(seed=6)

plt.figure(figsize=(6, 4))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train.squeeze(), cmap="coolwarm", s=32, edgecolor="black", linewidth=0.4)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Training data: small, noisy, nonlinear")
plt.grid(True, alpha=0.35)
plt.tight_layout()
plt.show()

In [ ]:
class MLPClassifier(torch.nn.Module):
    def __init__(
        self,
        hidden_width=32,
        n_hidden_layers=2,
        activation="relu",
        dropout=0.0,
        batch_norm=False,
        init="kaiming",
        init_scale=1.0,
    ):
        super().__init__()
        activation = activation.lower()
        layers = []
        in_features = 2

        for _ in range(n_hidden_layers):
            linear = torch.nn.Linear(in_features, hidden_width)
            layers.append(linear)
            if batch_norm:
                layers.append(torch.nn.BatchNorm1d(hidden_width))
            if activation == "relu":
                layers.append(torch.nn.ReLU())
            elif activation == "tanh":
                layers.append(torch.nn.Tanh())
            else:
                raise ValueError("activation must be 'relu' or 'tanh'")
            if dropout > 0:
                layers.append(torch.nn.Dropout(dropout))
            in_features = hidden_width

        layers.append(torch.nn.Linear(in_features, 1))
        self.net = torch.nn.Sequential(*layers)
        self.reset_parameters(init=init, activation=activation, init_scale=init_scale)

    def reset_parameters(self, init="kaiming", activation="relu", init_scale=1.0):
        for module in self.modules():
            if isinstance(module, torch.nn.Linear):
                if init == "kaiming":
                    torch.nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
                elif init == "xavier":
                    gain = torch.nn.init.calculate_gain("tanh" if activation == "tanh" else "relu")
                    torch.nn.init.xavier_normal_(module.weight, gain=gain)
                elif init == "normal":
                    torch.nn.init.normal_(module.weight, mean=0.0, std=init_scale)
                else:
                    raise ValueError("init must be 'kaiming', 'xavier', or 'normal'")
                torch.nn.init.zeros_(module.bias)

    def forward(self, X):
        return self.net(X)


def accuracy_from_logits(logits, y):
    preds = (torch.sigmoid(logits) >= 0.5).float()
    return float((preds == y).float().mean())


def train_model(
    config,
    lr=0.01,
    n_epochs=300,
    seed=42,
    weight_decay=0.0,
    grad_clip=None,
    early_stopping=False,
    patience_epochs=30,
):
    set_seed(seed)
    model = MLPClassifier(**config).to(device)
    loss_fn = torch.nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "grad_norm": []}
    best_state = copy.deepcopy(model.state_dict())
    best_val = float("inf")
    wait = 0

    for _ in range(n_epochs):
        model.train()
        logits = model(X_train)
        loss = loss_fn(logits, y_train)

        optimizer.zero_grad()
        loss.backward()

        total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip or 1e9)
        optimizer.step()

        model.eval()
        with torch.no_grad():
            train_logits = model(X_train)
            val_logits = model(X_val)
            train_loss = loss_fn(train_logits, y_train)
            val_loss = loss_fn(val_logits, y_val)

            history["train_loss"].append(float(train_loss))
            history["val_loss"].append(float(val_loss))
            history["train_acc"].append(accuracy_from_logits(train_logits, y_train))
            history["val_acc"].append(accuracy_from_logits(val_logits, y_val))
            history["grad_norm"].append(float(total_norm))

        if not np.isfinite(history["train_loss"][-1]) or not np.isfinite(history["val_loss"][-1]):
            break

        if early_stopping:
            if history["val_loss"][-1] < best_val:
                best_val = history["val_loss"][-1]
                best_state = copy.deepcopy(model.state_dict())
                wait = 0
            else:
                wait += 1
                if wait >= patience_epochs:
                    break

    if early_stopping:
        model.load_state_dict(best_state)

    return model, history


def plot_history(histories, title, metric="loss"):
    plt.figure(figsize=(8, 4))
    for label, history in histories:
        if metric == "loss":
            plt.plot(np.maximum(history["train_loss"], 1e-8), linestyle="-", label=f"{label} train")
            plt.plot(np.maximum(history["val_loss"], 1e-8), linestyle="--", label=f"{label} val")
            plt.ylabel("BCE loss")
            plt.yscale("log")
        elif metric == "accuracy":
            plt.plot(history["train_acc"], linestyle="-", label=f"{label} train")
            plt.plot(history["val_acc"], linestyle="--", label=f"{label} val")
            plt.ylabel("Accuracy")
        else:
            raise ValueError("metric must be 'loss' or 'accuracy'")
    plt.xlabel("Epoch")
    plt.title(title)
    plt.grid(True, which="both", alpha=0.35)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_decision_boundary(model, title):
    model.eval()
    x1 = np.linspace(-2.7, 2.7, 160)
    x2 = np.linspace(-2.7, 2.7, 160)
    xx, yy = np.meshgrid(x1, x2)
    grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()].astype(np.float32), device=device)
    with torch.no_grad():
        probs = torch.sigmoid(model(grid)).cpu().numpy().reshape(xx.shape)

    plt.figure(figsize=(6, 4))
    plt.contourf(xx, yy, probs, levels=20, cmap="coolwarm", alpha=0.75)
    plt.contour(xx, yy, probs, levels=[0.5], colors="black", linewidths=1.5)
    plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train.squeeze(), cmap="coolwarm", s=28, edgecolor="black", linewidth=0.4)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(title)
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

---

## 2. Diagnosis dashboard

We train three deliberately different models:

- **Too small:** not enough capacity to learn the curve.
- **Too large:** can memorize the small noisy training set.
- **Unstable:** training is made difficult with aggressive settings.

Read the curves first. The fix comes after the diagnosis.

In [ ]:
small_config = dict(hidden_width=4, n_hidden_layers=0, activation="tanh", dropout=0.0, batch_norm=False, init="xavier")
large_config = dict(hidden_width=128, n_hidden_layers=4, activation="relu", dropout=0.0, batch_norm=False, init="kaiming")
unstable_config = dict(hidden_width=64, n_hidden_layers=5, activation="tanh", dropout=0.0, batch_norm=False, init="normal", init_scale=1.8)

small_model, small_history = train_model(small_config, lr=0.01, n_epochs=350, seed=1)
large_model, large_history = train_model(large_config, lr=0.01, n_epochs=450, seed=1)
unstable_model, unstable_history = train_model(unstable_config, lr=0.03, n_epochs=120, seed=1)

plot_history(
    [
        ("too small", small_history),
        ("too large", large_history),
        ("unstable", unstable_history),
    ],
    "Diagnosis dashboard: different failure modes",
)

for label, history in [("too small", small_history), ("too large", large_history), ("unstable", unstable_history)]:
    print(f"{label:10s} train_acc={history['train_acc'][-1]:.3f} val_acc={history['val_acc'][-1]:.3f}")

**Pause and reflect**

1. Which model underfits?
2. Which model has the biggest train/validation gap?
3. Which model looks unstable from the curve or gradient behavior?

---

## 3. Capacity: underfitting vs overfitting

Model capacity is how flexible the model is. More capacity can help fit structure, but too much capacity can fit noise.

**Rule of thumb:** increase capacity when both train and validation are bad; reduce or regularize capacity when train is good but validation is bad.

### Intuition

Imagine drawing the decision boundary by hand.

- A tiny model can draw only a simple line or smooth bend. It may miss the real pattern.
- A medium model can follow the main curve.
- A very large model can start following accidental wiggles from the small noisy training set.

**Prediction before running:** Which model will have the best training accuracy? Which one will have the best validation accuracy?

In [ ]:
capacity_specs = [
    ("small", dict(hidden_width=4, n_hidden_layers=0, activation="tanh", dropout=0.0, batch_norm=False, init="xavier")),
    ("medium", dict(hidden_width=32, n_hidden_layers=2, activation="relu", dropout=0.0, batch_norm=False, init="kaiming")),
    ("large", dict(hidden_width=128, n_hidden_layers=4, activation="relu", dropout=0.0, batch_norm=False, init="kaiming")),
]

capacity_runs = []
for label, config in capacity_specs:
    model, history = train_model(config, lr=0.01, n_epochs=400, seed=4)
    capacity_runs.append((label, model, history))

plot_history([(label, history) for label, _, history in capacity_runs], "Capacity comparison")
plot_history([(label, history) for label, _, history in capacity_runs], "Capacity comparison - accuracy", metric="accuracy")

In [ ]:
for label, model, history in capacity_runs:
    plot_decision_boundary(model, f"Decision boundary: {label} model")

---

## 4. Coding block 1: diagnose capacity (about 40 min)

**Goal:** create and label examples of underfitting, reasonable fitting, and overfitting.

**Core path**

1. Edit `experiments`.
2. Keep the same dataset and seed.
3. Compare train loss, validation loss, train accuracy, and validation accuracy.
4. Label each run: underfit, reasonable, overfit, or unstable.

**Question:** Which change helped validation, not just training?

In [ ]:
# TODO: edit these experiments.
# Format: label, config, learning_rate, epochs
experiments = [
    ("tiny", dict(hidden_width=4, n_hidden_layers=0, activation="tanh", dropout=0.0, batch_norm=False, init="xavier"), 0.01, 300),
    ("reasonable", dict(hidden_width=32, n_hidden_layers=2, activation="relu", dropout=0.0, batch_norm=False, init="kaiming"), 0.01, 300),
    ("very large", dict(hidden_width=128, n_hidden_layers=4, activation="relu", dropout=0.0, batch_norm=False, init="kaiming"), 0.01, 450),
]

coding1_runs = []
for label, config, lr, epochs in experiments:
    model, history = train_model(config, lr=lr, n_epochs=epochs, seed=9)
    coding1_runs.append((label, model, history))

plot_history([(label, history) for label, _, history in coding1_runs], "Coding block 1: capacity experiments")

for label, _, history in coding1_runs:
    gap = history["train_acc"][-1] - history["val_acc"][-1]
    print(f"{label:12s} train_acc={history['train_acc'][-1]:.3f} val_acc={history['val_acc'][-1]:.3f} gap={gap:.3f}")

<details>
<summary>One possible diagnosis</summary>

- Tiny model: likely underfits.
- Reasonable model: usually gives a better train/validation balance.
- Very large model: may fit train data better but can increase the generalization gap.

</details>

---

## 5. Stability fixes

Training instability often comes from updates that are too large or gradients that explode.

Practical fixes:

- lower the learning rate
- use a more suitable initialization
- use an activation/init pair that makes sense
- clip gradients before the optimizer step

**Gradient clipping:** if the gradient norm is bigger than a threshold, rescale it before updating.

### Intuition

The optimizer is trying to walk downhill. If a gradient is extremely large, one update can jump far away from the useful region. Gradient clipping says: "you may move in this direction, but not with an unlimited step."

Initialization and activation matter too. ReLU networks are commonly paired with Kaiming initialization; Tanh networks are often paired with Xavier initialization. The goal is to keep signals and gradients in a useful range as they move through layers.

**Prediction before running:** Will clipping mainly improve train accuracy, validation accuracy, or the smoothness of the curve?

In [ ]:
bad_stability_config = dict(
    hidden_width=64,
    n_hidden_layers=5,
    activation="tanh",
    dropout=0.0,
    batch_norm=False,
    init="normal",
    init_scale=1.8,
)

better_stability_config = dict(
    hidden_width=64,
    n_hidden_layers=5,
    activation="relu",
    dropout=0.0,
    batch_norm=False,
    init="kaiming",
)

bad_model, bad_hist = train_model(bad_stability_config, lr=0.03, n_epochs=160, seed=3)
clipped_model, clipped_hist = train_model(bad_stability_config, lr=0.03, n_epochs=160, seed=3, grad_clip=1.0)
better_model, better_hist = train_model(better_stability_config, lr=0.01, n_epochs=160, seed=3, grad_clip=1.0)

plot_history(
    [
        ("bad init/high lr", bad_hist),
        ("same + clipping", clipped_hist),
        ("better init + clipping", better_hist),
    ],
    "Stability fixes",
)

plt.figure(figsize=(8, 3))
for label, history in [("bad", bad_hist), ("clipped", clipped_hist), ("better", better_hist)]:
    plt.plot(np.maximum(history["grad_norm"], 1e-8), label=label)
plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("Gradient norm")
plt.title("Gradient norms")
plt.grid(True, which="both", alpha=0.35)
plt.legend()
plt.tight_layout()
plt.show()

---

## 6. Regularization fixes: make memorization harder

Regularization reduces the model's ability to memorize noise.

### Dropout

During training, dropout randomly turns off some activations. The model cannot rely too heavily on one path.

**Intuition:** dropout makes the network practice with missing pieces. A memorized shortcut becomes less reliable, so the model is pushed toward more robust patterns.

### Batch normalization

Batch norm normalizes intermediate activations. It can make training smoother, especially in deeper networks.

**Intuition:** each layer receives inputs with a more predictable scale. This often makes optimization less jumpy.

### Early stopping

Early stopping watches validation loss and stops before the model keeps memorizing.

**Intuition:** the model may keep improving on training data after it has stopped improving on unseen data. Early stopping treats that validation turn as a signal to stop.

**Prediction before running:** Which fix might reduce training performance but improve validation performance?

In [ ]:
overfit_config = dict(hidden_width=128, n_hidden_layers=4, activation="relu", dropout=0.0, batch_norm=False, init="kaiming")
dropout_config = dict(hidden_width=128, n_hidden_layers=4, activation="relu", dropout=0.25, batch_norm=False, init="kaiming")
batchnorm_dropout_config = dict(hidden_width=128, n_hidden_layers=4, activation="relu", dropout=0.20, batch_norm=True, init="kaiming")

overfit_model, overfit_hist = train_model(overfit_config, lr=0.01, n_epochs=500, seed=8)
dropout_model, dropout_hist = train_model(dropout_config, lr=0.01, n_epochs=500, seed=8)
bn_do_model, bn_do_hist = train_model(batchnorm_dropout_config, lr=0.01, n_epochs=500, seed=8)
early_model, early_hist = train_model(batchnorm_dropout_config, lr=0.01, n_epochs=500, seed=8, early_stopping=True, patience_epochs=35)

plot_history(
    [
        ("overfit baseline", overfit_hist),
        ("dropout", dropout_hist),
        ("batchnorm+dropout", bn_do_hist),
        ("+ early stopping", early_hist),
    ],
    "Regularization fixes",
)

for label, history in [
    ("baseline", overfit_hist),
    ("dropout", dropout_hist),
    ("batchnorm+dropout", bn_do_hist),
    ("early stopping", early_hist),
]:
    print(f"{label:18s} epochs={len(history['train_loss']):3d} train_acc={history['train_acc'][-1]:.3f} val_acc={history['val_acc'][-1]:.3f}")

---

## Interactive repair lab

Optional live panel. Use this after the static examples above.

Turn repair tools on and off, press **Train / rerun**, and compare the train/validation curves. This is not required for submission; it is a quick way to build intuition.

**Watch for:** a fix can make training accuracy slightly worse while validation accuracy improves.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except Exception as exc:
    widgets = None
    print("Widgets are optional. If this import fails, continue with the static coding block below.")
    print(exc)

In [ ]:
if widgets is not None:
    width_slider = widgets.IntSlider(value=128, min=8, max=160, step=8, description="width")
    layers_slider = widgets.IntSlider(value=4, min=1, max=5, step=1, description="layers")
    lr_slider = widgets.FloatLogSlider(value=0.01, base=10, min=-3, max=-1, step=0.25, description="lr")
    dropout_check = widgets.Checkbox(value=True, description="dropout")
    dropout_slider = widgets.FloatSlider(value=0.20, min=0.0, max=0.5, step=0.05, description="dropout rate")
    batchnorm_check = widgets.Checkbox(value=True, description="batch norm")
    clipping_check = widgets.Checkbox(value=True, description="clip gradients")
    early_check = widgets.Checkbox(value=True, description="early stopping")
    show_baseline = widgets.Checkbox(value=True, description="show baseline")
    show_repair = widgets.Checkbox(value=True, description="show repair")
    train_button = widgets.Button(description="Train / rerun", button_style="primary")
    output = widgets.Output()

    controls = widgets.VBox([
        widgets.HBox([width_slider, layers_slider, lr_slider]),
        widgets.HBox([dropout_check, dropout_slider, batchnorm_check]),
        widgets.HBox([clipping_check, early_check]),
        widgets.HBox([show_baseline, show_repair, train_button]),
    ])

    def run_interactive_repair(_=None):
        with output:
            clear_output(wait=True)
            config = dict(
                hidden_width=width_slider.value,
                n_hidden_layers=layers_slider.value,
                activation="relu",
                dropout=dropout_slider.value if dropout_check.value else 0.0,
                batch_norm=batchnorm_check.value,
                init="kaiming",
            )
            repair_model, repair_history = train_model(
                config,
                lr=lr_slider.value,
                n_epochs=350,
                seed=15,
                grad_clip=1.0 if clipping_check.value else None,
                early_stopping=early_check.value,
                patience_epochs=30,
            )
            curves = []
            if show_baseline.value:
                curves.append(("baseline", overfit_hist))
            if show_repair.value:
                curves.append(("interactive repair", repair_history))
            plot_history(curves, "Interactive repair lab")
            print(f"repair epochs={len(repair_history['train_loss'])}")
            print(f"train_acc={repair_history['train_acc'][-1]:.3f}")
            print(f"val_acc={repair_history['val_acc'][-1]:.3f}")

    train_button.on_click(run_interactive_repair)
    display(controls, output)

---

## 7. Coding block 2: repair a broken model (about 40-45 min)

**Scenario:** the model below is too flexible and somewhat unstable. Your job is to repair it.

**Core path**

1. Run the baseline.
2. Change one repair setting at a time.
3. Compare train and validation curves.
4. Keep the repair that improves validation behavior.

Possible repair knobs:

- `dropout`
- `batch_norm`
- `learning_rate`
- `grad_clip`
- `early_stopping`
- `patience_epochs`
- `hidden_width` and `n_hidden_layers`

In [ ]:
# TODO: repair this model.
repair_config = dict(
    hidden_width=128,
    n_hidden_layers=4,
    activation="relu",
    dropout=0.20,       # TODO: try 0.0, 0.1, 0.25, 0.4
    batch_norm=True,    # TODO: try False vs True
    init="kaiming",
)

learning_rate = 0.01    # TODO: try 0.003, 0.01, 0.03
grad_clip = 1.0         # TODO: try None, 0.5, 1.0, 5.0
use_early_stopping = True
patience_epochs = 35

broken_model, broken_history = train_model(
    overfit_config,
    lr=0.01,
    n_epochs=500,
    seed=11,
    grad_clip=None,
    early_stopping=False,
)

repaired_model, repaired_history = train_model(
    repair_config,
    lr=learning_rate,
    n_epochs=500,
    seed=11,
    grad_clip=grad_clip,
    early_stopping=use_early_stopping,
    patience_epochs=patience_epochs,
)

plot_history(
    [
        ("broken baseline", broken_history),
        ("your repair", repaired_history),
    ],
    "Coding block 2: broken model vs repair",
)

print("Broken final val acc: ", broken_history["val_acc"][-1])
print("Repair final val acc: ", repaired_history["val_acc"][-1])
print("Repair epochs run:    ", len(repaired_history["train_loss"]))

In [ ]:
plot_decision_boundary(broken_model, "Broken baseline decision boundary")
plot_decision_boundary(repaired_model, "Repaired model decision boundary")

### Repair report prompt

Write 3-5 sentences:

1. What was the failure mode?
2. Which fix helped most?
3. What tradeoff did you notice?
4. Which plot supports your decision?

---

## 8. Assignment-style repair

For Assignment 1, the target skill is not "try every trick." The target skill is:

1. diagnose the failure
2. choose a small set of fixes
3. compare curves
4. justify the final model

### Minimum evidence

- train loss and validation loss plot
- train accuracy and validation accuracy
- short explanation of the chosen fixes
- one comparison against the broken baseline

The controlled assignment notebook is:

`Assignment_01_Model_Repair.ipynb`

Use this teaching notebook for exploration. Use the assignment notebook for the graded, comparable submission.

---

## Wrap-up: takeaways

1. Underfitting and overfitting are different problems with different fixes.
2. Capacity helps only if validation improves too.
3. Dropout, batch norm, gradient clipping, initialization, and early stopping are repair tools.
4. Do not stack fixes blindly. Change one thing, re-run, and compare.
5. Assignment 1 is about diagnosis plus justification, not only a high final score.

---

## Homework / Post-class Extensions

Optional unless assigned.

| Idea | What to try |
|------|-------------|
| Weight decay | Add `weight_decay` to Adam and compare with dropout |
| Data noise | Increase or decrease dataset noise and repeat diagnosis |
| Validation size | Change train/validation split sizes |
| Activation search | Compare ReLU and Tanh with matching init |
| Clip threshold search | Sweep `grad_clip` values |
| Repair grid | Try a small grid over dropout, batch norm, and learning rate |

### Extension A: weight decay

In [ ]:
# TODO: train the same model with weight_decay=0.0, 1e-4, 1e-3.
# Compare validation loss and validation accuracy.
pass

### Extension B: repair grid

In [ ]:
# TODO: loop over a small grid:
# dropout_values = [0.0, 0.1, 0.25]
# batch_norm_values = [False, True]
# learning_rates = [0.003, 0.01]
# Print the best validation accuracy.
pass

### Extension C: early stopping sensitivity

In [ ]:
# TODO: compare patience_epochs = 10, 30, 60.
# How does patience change training time and validation performance?
pass